# Chapter 8: Stakeholder Communication

**Book:** [LLMs for Business & Quality Analysts](../index.html)

## Lab Overview
Build a **Report Generator** that takes raw project data and generates tailored communications for different audiences -- executives, technical teams, and business stakeholders.

## Prerequisites
- Python 3.9+
- OpenAI API key
- Understanding of stakeholder management concepts


In [ ]:
# Setup
!pip install -q openai python-dotenv

import os
import json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o'

## 1. Project Status Data

Define raw project data that will be transformed into different communication formats.


In [ ]:
# Raw project status data
project_data = {
    'project': 'Customer Portal v2.0',
    'sprint': 'Sprint 14',
    'date': '2024-11-15',
    'velocity': {'planned': 34, 'completed': 28, 'carried_over': 6},
    'stories': {
        'total': 12, 'completed': 8, 'in_progress': 3, 'blocked': 1
    },
    'defects': {
        'critical': 1, 'high': 3, 'medium': 7, 'low': 12,
        'resolved_this_sprint': 15, 'new_this_sprint': 8
    },
    'blocked_item': {
        'story': 'STORY-245: SSO Integration with Okta',
        'reason': 'Waiting for Okta sandbox credentials from vendor',
        'days_blocked': 5,
        'impact': 'Delays authentication epic by 1 sprint'
    },
    'risks': [
        {'description': 'Third-party API rate limits may affect performance testing', 'probability': 'Medium', 'impact': 'High'},
        {'description': 'Key developer on vacation next sprint', 'probability': 'Certain', 'impact': 'Medium'}
    ],
    'milestones': {
        'beta_release': {'target': '2024-12-15', 'status': 'At Risk', 'confidence': '60%'},
        'ga_release': {'target': '2025-01-31', 'status': 'On Track', 'confidence': '80%'}
    }
}

print(json.dumps(project_data, indent=2))

## 2. Generate Audience-Specific Reports

Create tailored reports for different stakeholder audiences.


In [ ]:
def generate_report(data: dict, audience: str, format_style: str) -> str:
    """Generate a stakeholder report tailored to the audience."""
    audience_guides = {
        'executive': 'Focus on business impact, milestone status, and key decisions needed. Use bullet points. Keep under 200 words. Highlight risks in terms of revenue/timeline impact. No technical jargon.',
        'technical': 'Include velocity metrics, defect trends, technical blockers, and architecture concerns. Use specific numbers. Include sprint burndown insights.',
        'business': 'Focus on feature delivery, customer impact, and timeline. Explain blockers in business terms. Include what users will see next sprint.'
    }
    
    prompt = f"""Generate a {format_style} project status report for a {audience} audience.

Audience guidance: {audience_guides[audience]}

Project Data:
{json.dumps(data, indent=2)}

Include appropriate sections, formatting, and tone for the audience."""
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': f'You are a senior project communicator writing for {audience} stakeholders.'},
            {'role': 'user', 'content': prompt}
        ],
        temperature=0.4
    )
    return response.choices[0].message.content

# Generate executive summary
exec_report = generate_report(project_data, 'executive', 'concise email')
print('EXECUTIVE REPORT:')
print('=' * 60)
print(exec_report)

In [ ]:
# Generate technical team report
tech_report = generate_report(project_data, 'technical', 'detailed sprint review')
print('TECHNICAL TEAM REPORT:')
print('=' * 60)
print(tech_report)

In [ ]:
# Generate a risk escalation email
escalation_prompt = f"""Write a professional risk escalation email based on this project data.

The blocked item has been stuck for {project_data['blocked_item']['days_blocked']} days 
and threatens the beta release milestone.

Project Data: {json.dumps(project_data)}

The email should:
1. Clearly state the risk and its business impact
2. Provide context without blaming anyone
3. Request specific actions with deadlines
4. Offer alternatives if the blocker cannot be resolved
5. Set a follow-up date

Tone: professional, urgent but not panicked."""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': escalation_prompt}],
    temperature=0.3
)
print('ESCALATION EMAIL:')
print('=' * 60)
print(response.choices[0].message.content)

## Exercises
1. Generate a sprint retrospective summary from the project data.
2. Create a stakeholder-specific FAQ that anticipates questions each audience might ask.
3. Build a meeting agenda generator that creates structured agendas based on project status.
4. Add a tone-adjustment feature that can make the same report more or less urgent.
